# 4. óra · Klasszifikáció: a nyers adattól a hangolt modellig
**Adat:** Kaggle „Don't Get Kicked!” (Carvana; OpenML 41162, CC0): 72 983 használtautó-vásárlás egy aukción. **Címke:** `IsBadBuy` — rossz vétel („kick”), a vásárlások ~12%-a.
**Cél:** előre jelezni, melyik vásárlás lesz rossz vétel.

Ez a notebook a slide-ok számait és ábráit generálja (`assets/`). Minden szám innen jön, rögzített seedekkel. Az órán nem vetítjük.
Futtatás a lecke mappájából: `uv run jupyter nbconvert --to notebook --execute demo.ipynb --inplace`.

In [1]:
import json, os, time, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_predict, cross_val_score, cross_validate, RandomizedSearchCV)
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             average_precision_score, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve)
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import make_pipeline as imb_pipeline
from scipy.stats import randint, uniform, loguniform

# A deck színeihez igazítva (assets/deck/deck.css), mint a 3. órán: sötét háttér, validált sötét módú lépések.
SURFACE, INK, INK2, GRID = "#0d2a20", "#e9e4d2", "#b9c2b3", "#30483c"
BLUE, ORANGE = "#3987e5", "#d95926"          # jó vétel / rossz vétel
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.edgecolor": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.alpha": .7,
    "font.size": 15, "axes.titlesize": 17, "axes.labelsize": 15, "legend.fontsize": 14, "figure.dpi": 110})
DATA, OUT = Path("data/kick.csv.gz"), Path("assets"); OUT.mkdir(exist_ok=True)
hu = lambda v, nd=2: f"{v:.{nd}f}".replace(".", ",")                       # magyar tizedesvessző
thou = lambda v: f"{v:,.0f}".replace(",", " ")                              # ezres tagolás szóközzel
def hu_axes(ax, x=True, y=True):
    f = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))
    if x: ax.xaxis.set_major_formatter(f)
    if y: ax.yaxis.set_major_formatter(f)
def keep(fig, name): fig.savefig(OUT / f"{name}.png", dpi=170, bbox_inches="tight", facecolor=SURFACE); plt.close(fig)
NUMS = {}                                                                   # a slide-ok számai
SAMPLE_N = 30000        # az órához a 72 983 sorból egy 30 000 soros réteges mintát használunk (gyorsabb futás); None = teljes adat
_t0 = time.time()
def tick(msg):                                                              # haladásjelző (opcionális: NB_PROGRESS=<fájl>)
    if os.environ.get("NB_PROGRESS"): open(os.environ["NB_PROGRESS"], "a").write(f"{time.time()-_t0:6.0f}s  {msg}\n")

## 1. Nyers adat és EDA

In [2]:
full = pd.read_csv(DATA)
raw = full if SAMPLE_N is None else train_test_split(full, train_size=SAMPLE_N, stratify=full.IsBadBuy, random_state=0)[0].reset_index(drop=True)
NUMS.update(full_rows=len(full), full_bad_rate=float(full.IsBadBuy.mean()), sample_n=len(raw))
print(raw.shape, "(teljes adat:", full.shape, ")"); display(raw.head(3).T.head(34)); raw.info(); tick("betöltve")
y_all = raw.IsBadBuy.astype(int)
NUMS.update(n_rows=len(raw), n_cols=raw.shape[1], n_bad=int(y_all.sum()), bad_rate=float(y_all.mean()))
print(f"rossz vétel aránya: {y_all.mean():.1%}  -> a 'minden vétel jó' modell {1-y_all.mean():.1%}-ban igaza van")

(30000, 33) (teljes adat: (72983, 33) )


,0,1,2
IsBadBuy,0,0,0
PurchDate,1287532800,1287532800,1256083200
Auction,MANHEIM,OTHER,MANHEIM
VehYear,2008,2001,2006
VehicleAge,2,9,3
Make,DODGE,MERCURY,CHEVROLET
Model,CALIBER,SABLE,COBALT 2.2L I4 MPI
Trim,SE,LS,LT
SubModel,4D WAGON,4D SEDAN LS,4D SEDAN LT
Color,RED,GOLD,PURPLE


<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   IsBadBuy                           30000 non-null  int64  
 1   PurchDate                          30000 non-null  int64  
 2   Auction                            30000 non-null  str    
 3   VehYear                            30000 non-null  int64  
 4   VehicleAge                         30000 non-null  int64  
 5   Make                               30000 non-null  str    
 6   Model                              30000 non-null  str    
 7   Trim                               28982 non-null  str    
 8   SubModel                           29998 non-null  str    
 9   Color                              29998 non-null  str    
 10  Transmission                       29998 non-null  str    
 11  WheelTypeID                        28701 non-null  float64
 12  W

In [3]:
def bar_rates(ax, labels, rates, ns=None, color=BLUE, base=None, rot=0):
    ax.bar(range(len(labels)), rates, color=color, width=.62)
    for i, v in enumerate(rates): ax.text(i, v + .004, f"{v:.0%}", ha="center", fontsize=15, color=INK)
    if base is not None: ax.axhline(base, color=INK2, ls="--", lw=1.2)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=rot); ax.set_yticks([]); ax.grid(False)
    ax.set_ylim(0, max(rates) * 1.18)
age = raw[raw.VehicleAge.between(1, 9)]; ra = age.groupby("VehicleAge").IsBadBuy.mean()
fig, ax = plt.subplots(figsize=(9, 4)); bar_rates(ax, [str(i) for i in ra.index], ra.values, base=y_all.mean())
ax.set_xlabel("az autó kora a vásárláskor (év)"); ax.set_title(f"Rossz vétel aránya kor szerint (szaggatott: átlag, {y_all.mean():.0%})", loc="left"); keep(fig, "eda_kor")
NUMS["bad_by_age"] = ra.round(4).to_dict()
q = pd.qcut(raw.VehBCost, 8, duplicates="drop"); rp = raw.groupby(q, observed=True).IsBadBuy.mean()
fig, ax = plt.subplots(figsize=(10, 4)); bar_rates(ax, [f"≤{thou(iv.right)}" for iv in rp.index], rp.values, base=y_all.mean())
ax.set_xlabel("vételár (USD), nyolc egyenlő létszámú sáv"); ax.set_title("Rossz vétel aránya a vételár szerint", loc="left"); keep(fig, "eda_ar")
NUMS["bad_by_price_bin"] = {f"<={iv.right:.0f}": round(float(v), 4) for iv, v in rp.items()}
wt_all = raw.WheelType.fillna("hiányzik"); rw = raw.groupby(wt_all).IsBadBuy.agg(["mean", "size"]).loc[["Alloy", "Covers", "Special", "hiányzik"]]
fig, ax = plt.subplots(figsize=(9, 4.2)); bar_rates(ax, [f"{i}\n({thou(n)} autó)" for i, n in zip(rw.index, rw["size"])], rw["mean"].values, color=[BLUE, BLUE, BLUE, ORANGE], base=y_all.mean())
ax.set_title("Rossz vétel aránya a WheelType (kerék típusa) szerint", loc="left"); keep(fig, "eda_hianyzas")
NUMS["bad_by_wheeltype"] = {i: dict(rate=float(r), n=int(n)) for i, (r, n) in rw.iterrows()}
mi = raw.PRIMEUNIT.isna(); rm = raw.groupby(mi).IsBadBuy.mean()
NUMS.update(prime_missing=int(mi.sum()), bad_prime_missing=float(rm[True]), bad_prime_filled=float(rm[False]))

## 2. Adattisztítás és a split

In [4]:
mmr = [c for c in raw.columns if c.startswith("MMR")]
NUMS["zero_prices"] = {c: int((raw[c] == 0).sum()) for c in mmr}
print("nulla értékű árak (lehetetlen):", NUMS["zero_prices"])
print("Transmission:", raw.Transmission.value_counts(dropna=False).to_dict())
print("hiányzó értékek oszloponként:\n", raw.isna().sum()[lambda s: s > 0].to_string())
print("különböző értékek:", {c: int(raw[c].nunique()) for c in ["Make", "Model", "Trim", "SubModel", "BYRNO", "VNZIP1", "VNST"]})
t = pd.to_datetime(raw.PurchDate, unit="s"); print("vásárlás időszaka:", t.min().date(), "–", t.max().date())
NUMS.update(transmission_manual=int((raw.Transmission == "Manual").sum()), n_unique={c: int(raw[c].nunique()) for c in ["Model", "SubModel", "Trim", "VNZIP1", "BYRNO"]},
            n_missing={c: int(v) for c, v in raw.isna().sum()[lambda s: s > 0].items()}, date_min=str(t.min().date()), date_max=str(t.max().date()))

nulla értékű árak (lehetetlen): {'MMRAcquisitionAuctionAveragePrice': 348, 'MMRAcquisitionAuctionCleanPrice': 291, 'MMRAcquisitionRetailAveragePrice': 348, 'MMRAcquisitonRetailCleanPrice': 348, 'MMRCurrentAuctionAveragePrice': 209, 'MMRCurrentAuctionCleanPrice': 155, 'MMRCurrentRetailAveragePrice': 209, 'MMRCurrentRetailCleanPrice': 209}
Transmission: {'AUTO': 28929, 'MANUAL': 1068, nan: 2, 'Manual': 1}
hiányzó értékek oszloponként:
 Trim                                  1018
SubModel                                 2
Color                                    2
Transmission                             2
WheelTypeID                           1299
WheelType                             1299
MMRAcquisitionAuctionAveragePrice       10
MMRAcquisitionAuctionCleanPrice         10
MMRAcquisitionRetailAveragePrice        10
MMRAcquisitonRetailCleanPrice           10
MMRCurrentAuctionAveragePrice          135
MMRCurrentAuctionCleanPrice            135
MMRCurrentRetailAveragePrice           135
MMR

In [5]:
def clean(d):
    d = d.copy()
    d[mmr] = d[mmr].replace(0, np.nan)                                            # a 0 ár lehetetlen: hiányzó érték
    d["Transmission"] = d.Transmission.str.upper()                                # "Manual" vs "MANUAL"
    for c in ["WheelTypeID", "BYRNO", "VNZIP1", "IsOnlineSale"]:                  # számnak látszó KÓD, nem mennyiség
        d[c] = d[c].map(lambda v: np.nan if pd.isna(v) else str(int(v)))
    tt = pd.to_datetime(d.PurchDate, unit="s")                                    # unix időbélyeg -> dátumrészek
    d["purch_month"], d["purch_dow"] = tt.dt.month, tt.dt.dayofweek
    return d.drop(columns="PurchDate")
y = raw.IsBadBuy.astype(int); X = clean(raw.drop(columns="IsBadBuy"))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
# ---- X_test / y_test ZÁROLVA a 7. fejezetig. Addig minden keresztvalidáció a train halmazon. ----
NUMS.update(n_train=int(len(X_train)), n_test=int(len(X_test)), bad_train=float(y_train.mean()), bad_test=float(y_test.mean()), test_pos=int(y_test.sum()))
print(len(X_train), len(X_test), f"| rossz vétel train {y_train.mean():.3f} teszt {y_test.mean():.3f}")

# Két kódolás két modellcsaládnak: lineárisnak one-hot (+ skálázás, imputálás), fáknak elég a sorszám-kód
def prep_lin():
    return ColumnTransformer([("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="hiányzik"), OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30)), selector(dtype_exclude="number")),
                              ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), selector(dtype_include="number"))])
def prep_tree():
    return ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1), selector(dtype_exclude="number")),
                              ("num", "passthrough", selector(dtype_include="number"))])
cv = StratifiedKFold(3, shuffle=True, random_state=0)       # demó: 3 fold mindenhol (gyors); élesben 5-10
def xgb(n_jobs=8, **kw):
    p = dict(n_estimators=300, learning_rate=.05, max_depth=5, subsample=.8, colsample_bytree=.8, eval_metric="logloss", random_state=0, n_jobs=n_jobs); p.update(kw)
    return XGBClassifier(**p)

24000 6000 | rossz vétel train 0.123 teszt 0.123


In [6]:
def table_png(df, name, hl=None, fs=17):
    """Sötét témájú táblázat-ábra (a deck-motornak nincs táblázat-blokkja)."""
    widths = [max(len(str(v)) for v in [c, *df[c]]) * 0.155 + 0.8 for c in df.columns]
    fig, ax = plt.subplots(figsize=(sum(widths), .62 * (len(df) + 1))); ax.axis("off")
    t = ax.table(cellText=df.values.tolist(), colLabels=list(df.columns), loc="center", cellLoc="center", colWidths=[w / sum(widths) for w in widths])
    t.auto_set_font_size(False); t.set_fontsize(fs); t.scale(1, 1.9)
    for (r, c), cell in t.get_celld().items():
        cell.set_edgecolor(GRID); cell.set_linewidth(1)
        if r == 0: cell.set_facecolor("#123626"); cell.get_text().set_color("#d1a13a"); cell.get_text().set_fontweight("bold")
        else: cell.set_facecolor("#16442f" if hl and (r - 1) in hl else SURFACE); cell.get_text().set_color(INK)
    keep(fig, name)
# one-hot és sorszám-kód valódi sorokon (WheelType: a kerék típusa; hiányzik is lehet)
ex = X_train.drop_duplicates("WheelType").head(4)[["Make", "VehicleAge", "WheelType"]].reset_index(drop=True)
wt = ex.WheelType.fillna("hiányzik").to_frame("WheelType")
oh = OneHotEncoder(sparse_output=False).fit(X_train.WheelType.fillna("hiányzik").to_frame())
oh_df = pd.DataFrame(oh.transform(wt).astype(int), columns=[f"WheelType = {c}" for c in oh.categories_[0]])
table_png(pd.concat([ex[["Make", "VehicleAge"]].rename(columns={"Make": "márka", "VehicleAge": "kor"}), wt.rename(columns={"WheelType": "WheelType (eredeti)"}), oh_df], axis=1), "tbl_onehot", fs=16)
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1).fit(X_train[["WheelType"]])
table_png(pd.DataFrame({"WheelType (eredeti)": ex.WheelType.fillna("hiányzik"), "sorszám-kód (fáknak)": oe.transform(ex[["WheelType"]]).ravel().astype(int)}), "tbl_sorszam", fs=18)
NUMS["onehot_example"] = pd.concat([wt, oh_df], axis=1).to_dict("records"); NUMS["wheeltype_levels"] = [str(c) for c in oh.categories_[0]]
print("one-hot oszlopok száma a LINEÁRIS előfeldolgozás után:", prep_lin().fit(X_train).get_feature_names_out().shape[0], "| fáknál:", prep_tree().fit(X_train).get_feature_names_out().shape[0])
NUMS["n_cols_lin"] = int(prep_lin().fit(X_train).get_feature_names_out().shape[0]); NUMS["n_cols_tree"] = int(prep_tree().fit(X_train).get_feature_names_out().shape[0])

one-hot oszlopok száma a LINEÁRIS előfeldolgozás után: 643 | fáknál: 33


## 3. Keresztvalidáció és kiértékelés

In [7]:
# a keresztvalidáció (3-fold) szemléltetése
fig, ax = plt.subplots(figsize=(11, 3.6)); ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(-.4, 3.5)
for r in range(3):
    yy = 2.5 - r * 1.1
    ax.text(-.1, yy + .35, f"{r+1}. kör", ha="right", va="center", fontsize=17, color=INK)
    for c in range(3):
        isval = (c == r)
        ax.add_patch(Rectangle((c * 2.6 + .15, yy), 2.5, .8, facecolor=ORANGE if isval else BLUE, edgecolor=SURFACE, lw=2))
        ax.text(c * 2.6 + 1.4, yy + .4, "validáció" if isval else "tanítás", ha="center", va="center", fontsize=15, color="#0d2a20", fontweight="bold")
    ax.text(8.2, yy + .4, f"→ az {r+1}. rész jóslatai", va="center", fontsize=16, color=INK2)
ax.text(3.9, -.3, "az adat 3 egyenlő részre osztva; minden kör másik részt tart vissza", ha="center", fontsize=16, color=INK2)
keep(fig, "cv_abra")

In [8]:
dummy = make_pipeline(prep_tree(), DummyClassifier(strategy="most_frequent"))
pred = cross_val_predict(dummy, X_train, y_train, cv=cv)
NUMS["dummy"] = dict(acc=accuracy_score(y_train, pred), bal=balanced_accuracy_score(y_train, pred), recall=recall_score(y_train, pred))
print("'minden vétel jó':", {k: round(v, 3) for k, v in NUMS["dummy"].items()})
rng = np.random.default_rng(0); y_rare = (rng.random(10000) < .01).astype(int)
NUMS["rare_1pct"] = dict(acc=accuracy_score(y_rare, np.zeros(10000)), bal=balanced_accuracy_score(y_rare, np.zeros(10000)))
print("1%-os célváltozó, konstans 'negatív':", {k: round(v, 3) for k, v in NUMS["rare_1pct"].items()})

'minden vétel jó': {'acc': 0.877, 'bal': 0.5, 'recall': 0.0}
1%-os célváltozó, konstans 'negatív': {'acc': 0.991, 'bal': 0.5}


In [9]:
def cm_png(tn, fp, fn, tp, name):
    """Konfúziós mátrix, tényleges 2x2-es mátrixként."""
    fig, ax = plt.subplots(figsize=(9.5, 6.4)); ax.axis("off"); ax.set_xlim(-2.3, 4.4); ax.set_ylim(-.2, 4.5)
    cells = [(0, 0, tp, "TP", "megfogott rossz vétel", "#2f66a8"), (1, 0, fn, "FN", "kihagyott rossz vétel", "#a8431f"),
             (0, 1, fp, "FP", "fölösleges riasztás", "#a8431f"), (1, 1, tn, "TN", "jól hagyott jó vétel", "#2f66a8")]
    for cx, ry, val, tag, desc, col in cells:
        x0, y0 = cx * 2.2, 2.3 - ry * 1.9
        ax.add_patch(Rectangle((x0, y0), 2.1, 1.8, facecolor=col, edgecolor=SURFACE, lw=3))
        ax.text(x0 + 1.05, y0 + 1.22, thou(val), ha="center", va="center", fontsize=30, color="#f4f0e2", fontweight="bold")
        ax.text(x0 + 1.05, y0 + .72, tag, ha="center", va="center", fontsize=18, color="#f4f0e2", fontweight="bold")
        ax.text(x0 + 1.05, y0 + .33, desc, ha="center", va="center", fontsize=12.5, color="#f4f0e2")
    ax.text(1.05, 4.35, "Előrejelzés:\nrossz vétel", ha="center", va="center", fontsize=15, color=INK2); ax.text(3.25, 4.35, "Előrejelzés:\njó vétel", ha="center", va="center", fontsize=15, color=INK2)
    ax.text(-.15, 3.2, "Valóság:\nrossz vétel", ha="right", va="center", fontsize=15, color=INK2); ax.text(-.15, 1.3, "Valóság:\njó vétel", ha="right", va="center", fontsize=15, color=INK2)
    keep(fig, name)
lr = make_pipeline(prep_lin(), LogisticRegression(max_iter=3000))
p_lr = cross_val_predict(lr, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
tn, fp, fn, tp = confusion_matrix(y_train, p_lr > .5).ravel(); cm_png(tn, fp, fn, tp, "cm_lr")
NUMS["lr_confusion"] = dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn))
NUMS["lr_at_05"] = dict(precision=tp/(tp+fp), recall=tp/(tp+fn), spec=tn/(tn+fp), acc=accuracy_score(y_train, p_lr > .5), bal=balanced_accuracy_score(y_train, p_lr > .5))
print(NUMS["lr_confusion"], {k: round(v, 3) for k, v in NUMS["lr_at_05"].items()})
def thr_table(y, p, ts):
    return pd.DataFrame([{"küszöb": t, "megjelölt %": (p > t).mean() * 100, "precision": precision_score(y, p > t, zero_division=0), "recall": recall_score(y, p > t),
                          "kiegyens. pontosság": balanced_accuracy_score(y, p > t), "pontosság": accuracy_score(y, p > t)} for t in ts])
thr_lr = thr_table(y_train, p_lr, (.5, .3, .2, .15, .1)); display(thr_lr.round(3))
o = np.argsort(-p_lr); top = {}
for pct in (1, 5, 10, 20):
    k = int(len(o) * pct / 100); pr_k = float(y_train.values[o[:k]].mean()); top[pct] = dict(k=k, precision=pr_k, lift=pr_k / float(y_train.mean()))
NUMS["top_pct_lr"] = top; NUMS["lr_roc_auc"], NUMS["lr_pr_auc"] = float(roc_auc_score(y_train, p_lr)), float(average_precision_score(y_train, p_lr)); print(top, NUMS["lr_roc_auc"], NUMS["lr_pr_auc"])

{'TP': 740, 'FP': 265, 'FN': 2212, 'TN': 20783} {'precision': np.float64(0.736), 'recall': np.float64(0.251), 'spec': np.float64(0.987), 'acc': 0.897, 'bal': 0.619}


,küszöb,megjelölt %,precision,recall,kiegyens. pontosság,pontosság
0,0.50,4.188,0.736,0.251,0.619,0.897
1,0.30,8.075,0.538,0.353,0.655,0.883
2,0.20,14.688,0.382,0.456,0.676,0.842
3,0.15,22.287,0.302,0.547,0.685,0.789
4,0.10,36.733,0.231,0.690,0.684,0.679


{1: {'k': 240, 'precision': 0.875, 'lift': 7.1138211382113825}, 5: {'k': 1200, 'precision': 0.6791666666666667, 'lift': 5.521680216802168}, 10: {'k': 2400, 'precision': 0.47583333333333333, 'lift': 3.8685636856368566}, 20: {'k': 4800, 'precision': 0.32229166666666664, 'lift': 2.6202574525745255}} 0.756235521543737 0.44836038281193413


In [10]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5))
fpr_, tpr_, _ = roc_curve(y_train, p_lr); a.plot(fpr_, tpr_, color=BLUE, lw=2.5); a.plot([0, 1], [0, 1], color=INK2, ls="--", lw=1.2)
a.set_title(f"ROC-görbe, AUC = {hu(NUMS['lr_roc_auc'])}", loc="left"); a.set_xlabel("álriasztási arány (a jó vételek hányadát jelöltük meg)"); a.set_ylabel("recall (a rossz vételek hányadát fogtuk meg)")
pp_, rr_, _ = precision_recall_curve(y_train, p_lr); b.plot(rr_, pp_, color=BLUE, lw=2.5); b.axhline(y_train.mean(), color=INK2, ls="--", lw=1.2)
b.set_title(f"Precision-recall görbe, AP = {hu(NUMS['lr_pr_auc'])}", loc="left"); b.set_ylim(0, 1.02); b.set_xlabel("recall"); b.set_ylabel("precision")
b.text(.5, y_train.mean() + .03, f"alapvonal = {hu(y_train.mean())}", color=INK2, fontsize=14)
for ax_ in (a, b): hu_axes(ax_)
plt.tight_layout(w_pad=4); keep(fig, "metrics_roc_pr")

## 4. A modell-létra

In [11]:
models = {
    "dummy":              make_pipeline(prep_tree(), DummyClassifier(strategy="prior")),
    "logreg":             lr,
    "fa (korlát nélkül)": make_pipeline(prep_tree(), DecisionTreeClassifier(random_state=0)),
    "fa (mélység 6)":     make_pipeline(prep_tree(), DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, random_state=0)),
    "random forest":      make_pipeline(prep_tree(), RandomForestClassifier(300, min_samples_leaf=3, random_state=0, n_jobs=-1)),
    "xgboost":            make_pipeline(prep_tree(), xgb()),
}
scoring = {"PR-AUC": "average_precision", "ROC-AUC": "roc_auc", "bal_acc": "balanced_accuracy", "acc": "accuracy"}
lad, fold_pr = [], {}
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1); fold_pr[name] = r["test_PR-AUC"]
    lad.append({"modell": name, **{k: float(r["test_" + k].mean()) for k in scoring}, **{k + "_sd": float(r["test_" + k].std()) for k in scoring}})
lad = pd.DataFrame(lad); display(lad.round(3)); NUMS["ladder"] = lad.round(4).to_dict("records"); tick("létra kész")

,modell,PR-AUC,ROC-AUC,bal_acc,acc,PR-AUC_sd,ROC-AUC_sd,bal_acc_sd,acc_sd
0,dummy,0.123,0.500,0.500,0.877,0.000,0.000,0.000,0.000
1,logreg,0.449,0.756,0.619,0.897,0.008,0.006,0.005,0.002
2,fa (korlát nélkül),0.187,0.620,0.620,0.821,0.002,0.003,0.003,0.001
3,fa (mélység 6),0.412,0.732,0.609,0.898,0.008,0.011,0.006,0.000
4,random forest,0.462,0.755,0.622,0.900,0.011,0.010,0.003,0.001
5,xgboost,0.484,0.770,0.623,0.902,0.014,0.008,0.002,0.001


In [12]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
for i, (n_, s_) in enumerate(fold_pr.items()):
    ax.scatter(s_, [i] * len(s_), color=BLUE, s=70, alpha=.9, edgecolor=SURFACE); ax.hlines(i, s_.min(), s_.max(), color=BLUE, lw=1.5, alpha=.5)
ax.set_yticks(range(len(fold_pr))); ax.set_yticklabels(fold_pr); ax.invert_yaxis(); ax.set_xlabel("PR-AUC a 3 CV-fold mindegyikén"); hu_axes(ax, y=False); keep(fig, "ladder_folds")
xg_fit = models["xgboost"].fit(X_train, y_train)
imp = pd.Series(xg_fit[-1].feature_importances_, index=[n.split("__", 1)[-1] for n in xg_fit[0].get_feature_names_out()]).sort_values(ascending=False).head(10)[::-1]
fig, ax = plt.subplots(figsize=(9.5, 5)); ax.barh(imp.index, imp.values, color=BLUE, height=.6); ax.grid(False); ax.set_xlabel("fontosság az XGBoostban (gain)"); hu_axes(ax, y=False); keep(fig, "xgb_fontossag")
NUMS["xgb_top_features"] = imp[::-1].round(4).to_dict()

## 5. Visszacsatolási kör: hibák elemzése

In [13]:
loop_model = models["xgboost"]
p_oof = cross_val_predict(loop_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
def metrics(y, p, t=.5):
    return {"accuracy": accuracy_score(y, p > t), "balanced acc": balanced_accuracy_score(y, p > t), "recall": recall_score(y, p > t),
            "precision": precision_score(y, p > t, zero_division=0), "PR-AUC": average_precision_score(y, p)}
log = {"alap xgboost @0,5": metrics(y_train, p_oof)}
tn, fp, fn, tp = confusion_matrix(y_train, p_oof > .5).ravel(); NUMS["xgb_confusion"] = dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn)); cm_png(tn, fp, fn, tp, "cm_xgb")
print(NUMS["xgb_confusion"], {k: round(v, 3) for k, v in log["alap xgboost @0,5"].items()})
# néhány konkrét tévedés: a legbiztosabb álriasztások és a legnagyobb "szem elől tévesztések"
dr = X_train.MMRCurrentAuctionAveragePrice / X_train.MMRAcquisitionAuctionAveragePrice
d_ = X_train.assign(bad=y_train.values, p=p_oof, drift=dr)
fnr, fpr = d_[d_.bad == 1].nsmallest(5, "p"), d_[d_.bad == 0].nlargest(5, "p")
def fmt_rows(dd, kind): return pd.DataFrame({"tévedés": kind, "márka": dd.Make.values, "kor": dd.VehicleAge.values, "km": [thou(v) for v in dd.VehOdo], "vételár": [thou(v) for v in dd.VehBCost],
                                              "árváltozás": [hu(v, 2) if pd.notna(v) else "-" for v in dd.drift], "p(rossz)": [hu(v, 2) for v in dd.p]})
table_png(pd.concat([fmt_rows(fnr, "kihagyott rossz vétel"), fmt_rows(fpr, "álriasztás")]), "tbl_tevedesek", hl=[0, 1, 2, 3, 4], fs=15)
NUMS["fn_examples"] = fmt_rows(fnr, "FN").to_dict("records"); NUMS["fp_examples"] = fmt_rows(fpr, "FP").to_dict("records")
# AZ ÖTLET: a rossz vétel = elértéktelenedés. Mennyit mozdult az ár a vásárlás óta, és mit gondol a modell?
bins = pd.qcut(dr, 10, duplicates="drop")
g = pd.DataFrame({"y": y_train.values, "p": p_oof, "b": bins.values}).groupby("b", observed=True).agg(actual=("y", "mean"), predicted=("p", "mean"), n=("y", "size"))
labels = [("≤" if i == 0 else "") + hu(iv.right, 3) for i, iv in enumerate(g.index)]; labels[-1] = "≥" + hu(g.index[-1].left, 3)
fig, ax = plt.subplots(figsize=(11.5, 4.8)); x = np.arange(len(g)); w = .38
ax.bar(x - w/2, g.actual, w, color=ORANGE, label="valójában rossz vétel"); ax.bar(x + w/2, g.predicted, w, color=BLUE, label="a modell átlagos jóslata")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11.5); ax.set_xlabel("árváltozás a vásárlás óta: mai / vételkori aukciós átlagár (10 egyenlő létszámú sáv)")
ax.set_ylabel("rossz vétel aránya"); ax.legend(frameon=False, loc="upper right"); ax.grid(axis="x", visible=False); hu_axes(ax, x=False); keep(fig, "loop_arvaltozas")
NUMS["drift_gap"] = [dict(bin=l, actual=float(a), predicted=float(p_), n=int(n)) for l, a, p_, n in zip(labels, g.actual, g.predicted, g.n)]
print(g.round(3))

{'TP': 749, 'FP': 155, 'FN': 2203, 'TN': 20893} {'accuracy': 0.902, 'balanced acc': 0.623, 'recall': 0.254, 'precision': 0.829, 'PR-AUC': 0.483}


                actual  predicted     n
b                                      
(0.077, 0.88]    0.097      0.109  2360
(0.88, 0.926]    0.090      0.106  2360
(0.926, 0.957]   0.088      0.101  2359
(0.957, 0.984]   0.122      0.109  2360
(0.984, 1.0]     0.195      0.172  4314
(1.0, 1.005]     0.215      0.141   405
(1.005, 1.031]   0.125      0.112  2359
(1.031, 1.068]   0.112      0.106  2360
(1.068, 1.129]   0.106      0.106  2359
(1.129, 5.412]   0.098      0.114  2360


In [14]:
ts = np.arange(.1, .9, .01)
t_acc = max(ts, key=lambda t: ((p_oof > t) == y_train).mean())                         # 1. HIBA: pontosságra optimalizálunk
log["1. HIBA: küszöb a pontosságra"] = metrics(y_train, p_oof, t_acc)
Xo, yo = RandomOverSampler(random_state=0).fit_resample(X_train, y_train)               # 2. HIBA: előbb felülmintavételezünk ...
p_leaky = cross_val_predict(models["xgboost"], Xo, yo, cv=cv, method="predict_proba")[:, 1]   # ... és csak utána keresztvalidálunk
leaky = metrics(yo, p_leaky)
key = pd.util.hash_pandas_object(Xo, index=False).values
twin = {nm: float(np.mean([np.isin(key[v][yo.values[v] == lab], key[t]).mean() for t, v in cv.split(Xo, yo)])) for lab, nm in [(1, "rossz vételek"), (0, "jó vételek")]}
NUMS["leak"] = dict(rows=int(len(Xo)), pos=float(yo.mean()), twin=twin, leaky=leaky, t_acc=float(t_acc))
fixed = imb_pipeline(prep_tree(), RandomOverSampler(random_state=0), xgb())
p_fix = cross_val_predict(fixed, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
log["felülmintavételezés a foldon belül"] = metrics(y_train, p_fix)
print("pontosság-optimális küszöb", round(t_acc, 2), "| ikerpéldány arány:", {k: round(v, 3) for k, v in twin.items()}); display(pd.DataFrame({**log, "szivárgó (félrevezető)": leaky}).T.round(3)); tick("hibák (küszöb, szivárgás) kész")

pontosság-optimális küszöb 0.56 | ikerpéldány arány: {'rossz vételek': 0.992, 'jó vételek': 0.0}


,accuracy,balanced acc,recall,precision,PR-AUC
"alap xgboost @0,5",0.902,0.623,0.254,0.829,0.483
1. HIBA: küszöb a pontosságra,0.902,0.619,0.244,0.867,0.483
felülmintavételezés a foldon belül,0.816,0.682,0.504,0.334,0.475
szivárgó (félrevezető),0.808,0.808,0.777,0.829,0.899


In [15]:
def price_features(d):
    d = d.copy()
    d["drift_auc_avg"] = d.MMRCurrentAuctionAveragePrice / d.MMRAcquisitionAuctionAveragePrice
    d["drift_auc_clean"] = d.MMRCurrentAuctionCleanPrice / d.MMRAcquisitionAuctionCleanPrice
    d["drift_ret_avg"] = d.MMRCurrentRetailAveragePrice / d.MMRAcquisitionRetailAveragePrice
    d["drift_ret_clean"] = d.MMRCurrentRetailCleanPrice / d.MMRAcquisitonRetailCleanPrice
    d["drift_abs"] = d.MMRCurrentAuctionAveragePrice - d.MMRAcquisitionAuctionAveragePrice
    d["paid_vs_cur"] = d.VehBCost / d.MMRCurrentAuctionAveragePrice
    d["cur_spread"] = d.MMRCurrentRetailAveragePrice / d.MMRCurrentAuctionAveragePrice
    d["clean_vs_avg"] = d.MMRAcquisitionAuctionCleanPrice / d.MMRAcquisitionAuctionAveragePrice
    return d.replace([np.inf, -np.inf], np.nan)
ALL8 = ["drift_auc_avg", "drift_auc_clean", "drift_ret_avg", "drift_ret_clean", "drift_abs", "paid_vs_cur", "cur_spread", "clean_vs_avg"]
FOUR = ["drift_auc_avg", "drift_auc_clean", "drift_ret_avg", "drift_ret_clean"]
base_cols = list(X_train.columns); Xf_train_full = price_features(X_train)
rcv = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=1)
pr_cv = lambda m, Xf: cross_val_score(m, Xf, y_train, cv=rcv, scoring="average_precision", n_jobs=1)
bx, bl = pr_cv(models["xgboost"], X_train), pr_cv(lr, X_train)
NUMS["xgb_minus_lr_paired"] = dict(mean=float((bx - bl).mean()), sd=float((bx - bl).std()), n_folds=int(len(bx)))
feat_rows = [{"változat": "alap (nincs új jellemző)", "XGB PR-AUC": bx.mean(), "XGB páros különbség": np.nan, "XGB sd": np.nan, "LR PR-AUC": bl.mean(), "LR páros különbség": np.nan, "LR sd": np.nan}]
for nm, cols in [("+ árkülönbség (drift_abs)", ["drift_abs"]), ("+ 4 árarány (drift_*)", FOUR), ("+ mind a 8 ár-jellemző", ALL8)]:
    Xf = Xf_train_full[base_cols + cols]; sx, sl = pr_cv(models["xgboost"], Xf), pr_cv(lr, Xf); dx, dl = sx - bx, sl - bl
    feat_rows.append({"változat": nm, "XGB PR-AUC": sx.mean(), "XGB páros különbség": dx.mean(), "XGB sd": dx.std(), "LR PR-AUC": sl.mean(), "LR páros különbség": dl.mean(), "LR sd": dl.std()})
feat = pd.DataFrame(feat_rows); display(feat.round(4)); NUMS["features"] = feat.round(4).to_dict("records"); tick("jellemző-összehasonlítás kész")
X_train_fe = Xf_train_full[base_cols + ALL8]; X_test_fe = price_features(X_test)[base_cols + ALL8]
xgb_fe = make_pipeline(prep_tree(), xgb()); lr_fe = make_pipeline(prep_lin(), LogisticRegression(max_iter=3000))
p_fe = cross_val_predict(xgb_fe, X_train_fe, y_train, cv=cv, method="predict_proba")[:, 1]
log["jellemzők: árváltozás (8 új) @0,5"] = metrics(y_train, p_fe)
t_star = max(ts, key=lambda t: balanced_accuracy_score(y_train, p_fe > t))             # a VALÓDI javítás második fele: a működési pont
log["jellemzők + küszöb a kiegyens. pontosságra"] = metrics(y_train, p_fe, t_star)
thr_xgb = thr_table(y_train, p_fe, (.5, .3, t_star, .15, .1)); display(pd.DataFrame(log).T.round(3)); display(thr_xgb.round(3))
NUMS["loop"] = {k: {m: float(v) for m, v in d.items()} for k, d in log.items()}; NUMS["t_star"] = float(t_star)
NUMS["thr_xgb"] = thr_xgb.round(4).to_dict("records"); NUMS["thr_lr"] = thr_lr.round(4).to_dict("records")
dr_fe = X_train_fe.drift_auc_avg; g2 = pd.DataFrame({"y": y_train.values, "p": p_fe, "b": pd.qcut(dr_fe, 10, duplicates="drop").values}).groupby("b", observed=True).agg(actual=("y", "mean"), predicted=("p", "mean"))
NUMS["drift_gap_fe"] = [dict(actual=float(a), predicted=float(p_)) for a, p_ in zip(g2.actual, g2.predicted)]

,változat,XGB PR-AUC,XGB páros különbség,XGB sd,LR PR-AUC,LR páros különbség,LR sd
0,alap (nincs új jellemző),0.4779,NaN,NaN,0.4461,NaN,NaN
1,+ árkülönbség (drift_abs),0.4922,0.0143,0.0057,0.4455,-0.0005,0.0005
2,+ 4 árarány (drift_*),0.4934,0.0156,0.0060,0.4464,0.0003,0.0005
3,+ mind a 8 ár-jellemző,0.4958,0.0179,0.0062,0.4467,0.0006,0.0005


,accuracy,balanced acc,recall,precision,PR-AUC
"alap xgboost @0,5",0.902,0.623,0.254,0.829,0.483
1. HIBA: küszöb a pontosságra,0.902,0.619,0.244,0.867,0.483
felülmintavételezés a foldon belül,0.816,0.682,0.504,0.334,0.475
"jellemzők: árváltozás (8 új) @0,5",0.902,0.619,0.245,0.848,0.496
jellemzők + küszöb a kiegyens. pontosságra,0.753,0.698,0.624,0.277,0.496


,küszöb,megjelölt %,precision,recall,kiegyens. pontosság,pontosság
0,0.50,3.554,0.848,0.245,0.619,0.902
1,0.30,6.629,0.617,0.333,0.652,0.893
2,0.12,27.704,0.277,0.624,0.698,0.753
3,0.15,20.017,0.330,0.537,0.692,0.809
4,0.10,34.529,0.245,0.689,0.696,0.701


## 6. Hiperparaméter-hangolás

In [16]:
xgb_space = {"xgbclassifier__n_estimators": randint(100, 500), "xgbclassifier__learning_rate": loguniform(.01, .2), "xgbclassifier__max_depth": randint(3, 8),
             "xgbclassifier__min_child_weight": randint(1, 40), "xgbclassifier__subsample": uniform(.6, .4), "xgbclassifier__colsample_bytree": uniform(.4, .6),
             "xgbclassifier__reg_lambda": loguniform(.1, 20)}
rf_space = {"randomforestclassifier__max_depth": randint(6, 24), "randomforestclassifier__min_samples_leaf": randint(2, 60), "randomforestclassifier__max_features": uniform(.15, .6)}
search_models = {"xgboost": (make_pipeline(prep_tree(), xgb(n_jobs=1)), xgb_space, 8),
                 "random forest": (make_pipeline(prep_tree(), RandomForestClassifier(100, random_state=0, n_jobs=1)), rf_space, 3)}
searches = {}
for name, (m, space, n_iter) in search_models.items():
    searches[name] = RandomizedSearchCV(m, space, n_iter=n_iter, scoring="average_precision", cv=cv, random_state=0, n_jobs=8).fit(X_train_fe, y_train); tick(f"keresés kész: {name}")
    print(name, "legjobb CV PR-AUC", round(searches[name].best_score_, 4), {k.split("__")[1]: round(float(v), 3) for k, v in searches[name].best_params_.items()})
fresh = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=7)               # foldok, amiket a keresés sosem látott
defaults = {"logreg": lr_fe, "random forest": make_pipeline(prep_tree(), RandomForestClassifier(300, min_samples_leaf=3, random_state=0, n_jobs=-1)), "xgboost": xgb_fe}
tune = {}
for name in ["logreg", "random forest", "xgboost"]:
    s0 = cross_val_score(defaults[name], X_train_fe, y_train, cv=fresh, scoring="average_precision", n_jobs=1)
    s1 = s0 if name == "logreg" else cross_val_score(searches[name].best_estimator_, X_train_fe, y_train, cv=fresh, scoring="average_precision", n_jobs=8)
    tune[name] = dict(default=float(s0.mean()), default_sd=float(s0.std()), tuned=float(s1.mean()), tuned_sd=float(s1.std()), search_best=None if name == "logreg" else float(searches[name].best_score_))
display(pd.DataFrame(tune).T.round(4)); NUMS["tune"] = tune
NUMS["best_params_xgb"] = {k.split("__")[1]: float(v) for k, v in searches["xgboost"].best_params_.items()}
fig, ax = plt.subplots(figsize=(10, 4.6)); ys = np.arange(len(tune))
ax.errorbar([v["default"] for v in tune.values()], ys + .13, xerr=[v["default_sd"] for v in tune.values()], fmt="o", color=INK2, ecolor=INK2, ms=11, capsize=5, label="alapértelmezett")
ax.errorbar([v["tuned"] for v in tune.values()], ys - .13, xerr=[v["tuned_sd"] for v in tune.values()], fmt="o", color=BLUE, ecolor=BLUE, ms=11, capsize=5, label="hangolt")
ax.set_yticks(ys); ax.set_yticklabels(tune); ax.invert_yaxis(); ax.set_xlabel("PR-AUC új CV-foldokon (átlag ± szórás)"); ax.legend(frameon=False, loc="lower right"); hu_axes(ax, y=False); keep(fig, "tuning_elotte_utana")

xgboost legjobb CV PR-AUC 0.4953 {'colsample_bytree': 0.464, 'learning_rate': 0.041, 'max_depth': 6.0, 'min_child_weight': 29.0, 'n_estimators': 390.0, 'reg_lambda': 0.205, 'subsample': 0.73}


random forest legjobb CV PR-AUC 0.4696 {'max_depth': 18.0, 'max_features': 0.506, 'min_samples_leaf': 2.0}


,default,default_sd,tuned,tuned_sd,search_best
logreg,0.4502,0.0109,0.4502,0.0109,NaN
random forest,0.4726,0.0129,0.4675,0.0150,0.4696
xgboost,0.4991,0.0098,0.4932,0.0135,0.4953


## 7. Záró kiértékelés (a teszthalmaz, egyszer)

In [17]:
cands = {"logreg": lr_fe, "xgboost (alap)": xgb_fe, "xgboost (hangolt)": searches["xgboost"].best_estimator_}   # a szabály MINDEN jelöltet összevet
cv_scores = {n: float(cross_val_score(m, X_train_fe, y_train, cv=fresh, scoring="average_precision", n_jobs=1).mean()) for n, m in cands.items()}
final_name = max(cv_scores, key=cv_scores.get); print("CV PR-AUC:", {k: round(v, 4) for k, v in cv_scores.items()}, "-> végső:", final_name)
p_oof_final = cross_val_predict(cands[final_name], X_train_fe, y_train, cv=cv, method="predict_proba")[:, 1]
t_final = max(ts, key=lambda t: balanced_accuracy_score(y_train, p_oof_final > t)); print("küszöb a train OOF-ból:", round(t_final, 2))
test_p = {n: m.fit(X_train_fe, y_train).predict_proba(X_test_fe)[:, 1] for n, m in cands.items()}; p_te = test_p[final_name]
res = pd.DataFrame({"teszt @0,5": metrics(y_test, p_te, .5), f"teszt @{hu(t_final)}": metrics(y_test, p_te, t_final)}).T; display(res.round(3))
tn, fp, fn, tp = confusion_matrix(y_test, p_te > t_final).ravel(); cm_png(tn, fp, fn, tp, "cm_vegso")
rng = np.random.default_rng(0); yt = y_test.values; n = len(yt); idx = [rng.integers(0, n, n) for _ in range(1000)]
ap = lambda p, i: average_precision_score(yt[i], p[i])
ci = np.percentile([ap(p_te, i) for i in idx], [2.5, 97.5]); other = "logreg" if final_name != "logreg" else "xgboost (alap)"
diff = np.array([ap(p_te, i) - ap(test_p[other], i) for i in idx])
NUMS["final"] = dict(final_name=final_name, t_final=float(t_final), cv_scores=cv_scores, test_n=int(n), test_pos=int(yt.sum()),
    test_at_05=metrics(y_test, p_te, .5), test_at_t=metrics(y_test, p_te, t_final), confusion=dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn)),
    test_pr_auc={k: float(average_precision_score(y_test, v)) for k, v in test_p.items()}, boot_ci=[float(v) for v in ci],
    paired=dict(vs=other, mean=float(diff.mean()), lo=float(np.percentile(diff, 2.5)), hi=float(np.percentile(diff, 97.5))))
print(json.dumps(NUMS["final"], indent=1)); tick("záró teszt kész")
fig, ax = plt.subplots(figsize=(6.8, 5)); pp_, rr_, _ = precision_recall_curve(y_test, p_te); ax.plot(rr_, pp_, color=BLUE, lw=2.5); ax.axhline(y_test.mean(), color=INK2, ls="--", lw=1.2)
ax.scatter([recall_score(y_test, p_te > t_final)], [precision_score(y_test, p_te > t_final)], color=ORANGE, s=150, zorder=3, label=f"választott küszöb: {hu(t_final)}")
ax.set_ylim(0, 1.02); ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.legend(frameon=False, loc="upper right"); hu_axes(ax); keep(fig, "final_pr_gorbe")

CV PR-AUC: {'logreg': 0.4502, 'xgboost (alap)': 0.4991, 'xgboost (hangolt)': 0.4932} -> végső: xgboost (alap)


küszöb a train OOF-ból: 0.12


,accuracy,balanced acc,recall,precision,PR-AUC
"teszt @0,5",0.900,0.606,0.215,0.874,0.472
"teszt @0,12",0.751,0.692,0.615,0.273,0.472


{
 "final_name": "xgboost (alap)",
 "t_final": 0.12,
 "cv_scores": {
  "logreg": 0.4502291989023533,
  "xgboost (alap)": 0.49906463191013445,
  "xgboost (hangolt)": 0.49324522236053664
 },
 "test_n": 6000,
 "test_pos": 738,
 "test_at_05": {
  "accuracy": 0.8996666666666666,
  "balanced acc": 0.605538096429995,
  "recall": 0.21544715447154472,
  "precision": 0.8736263736263736,
  "PR-AUC": 0.4719183749738284
 },
 "test_at_t": {
  "accuracy": 0.7506666666666667,
  "balanced acc": 0.6924227395067566,
  "recall": 0.6151761517615176,
  "precision": 0.2725090036014406,
  "PR-AUC": 0.4719183749738284
 },
 "confusion": {
  "TP": 454,
  "FP": 1212,
  "FN": 284,
  "TN": 4050
 },
 "test_pr_auc": {
  "logreg": 0.40625909974055063,
  "xgboost (alap)": 0.4719183749738284,
  "xgboost (hangolt)": 0.4589244599937007
 },
 "boot_ci": [
  0.43860376018877895,
  0.5056973761085447
 ],
 "paired": {
  "vs": "logreg",
  "mean": 0.0647740154790373,
  "lo": 0.046402337192112536,
  "hi": 0.08464487182866935
 }
}

## Ábrák és táblázatok a slide-okhoz

In [18]:
f3 = lambda v: hu(v, 3)
t = lad.copy(); table_png(pd.DataFrame({"modell": t.modell, "PR-AUC": [f"{f3(a)} ± {f3(b)}" for a, b in zip(t["PR-AUC"], t["PR-AUC_sd"])],
    "ROC-AUC": [f"{f3(a)} ± {f3(b)}" for a, b in zip(t["ROC-AUC"], t["ROC-AUC_sd"])], "kiegyens. pont. @0,5": [f"{f3(a)} ± {f3(b)}" for a, b in zip(t.bal_acc, t.bal_acc_sd)],
    "pontosság @0,5": [f"{f3(a)} ± {f3(b)}" for a, b in zip(t.acc, t.acc_sd)]}), "tbl_letra")
table_png(pd.DataFrame({"küszöb": [hu(v, 2) for v in thr_lr["küszöb"]], "megjelölt": [f"{hu(v, 1)}%" for v in thr_lr["megjelölt %"]], "precision": [f3(v) for v in thr_lr.precision],
    "recall": [f3(v) for v in thr_lr.recall], "kiegyens. pont.": [f3(v) for v in thr_lr["kiegyens. pontosság"]], "pontosság": [f3(v) for v in thr_lr["pontosság"]]}), "tbl_kuszob_lr")
lg = pd.DataFrame(log).T
def loop_tbl(rows, name, hl=None):
    d = lg.loc[rows]; table_png(pd.DataFrame({"": rows, "pontosság": [f3(v) for v in d.accuracy], "kiegyens. pont.": [f3(v) for v in d["balanced acc"]], "recall": [f3(v) for v in d.recall],
        "precision": [f3(v) for v in d.precision], "PR-AUC": [f3(v) for v in d["PR-AUC"]]}), name, hl=hl)
loop_tbl(["alap xgboost @0,5", "1. HIBA: küszöb a pontosságra"], "tbl_hiba1")
lk = pd.DataFrame({"alap xgboost @0,5": lg.loc["alap xgboost @0,5"], "szivárgó: előbb mintavétel, aztán CV": pd.Series(leaky), "mintavétel a foldon belül": lg.loc["felülmintavételezés a foldon belül"]}).T
table_png(pd.DataFrame({"": lk.index, "pontosság": [f3(v) for v in lk.accuracy], "kiegyens. pont.": [f3(v) for v in lk["balanced acc"]], "recall": [f3(v) for v in lk.recall],
    "precision": [f3(v) for v in lk.precision], "PR-AUC": [f3(v) for v in lk["PR-AUC"]]}), "tbl_hiba2", hl=[1])
loop_tbl(list(lg.index[[0, 1, 2, 3, 4]]), "tbl_utazas", hl=[3, 4])
fe_t = feat.copy(); sg = lambda a, b: "-" if pd.isna(a) else f"{'+' if a >= 0 else '−'}{hu(abs(a), 4)} ± {hu(b, 4)}"
table_png(pd.DataFrame({"változat": fe_t["változat"], "XGB PR-AUC": [f3(v) for v in fe_t["XGB PR-AUC"]], "XGB páros különbség": [sg(a, b) for a, b in zip(fe_t["XGB páros különbség"], fe_t["XGB sd"])],
    "LR páros különbség": [sg(a, b) for a, b in zip(fe_t["LR páros különbség"], fe_t["LR sd"])]}), "tbl_jellemzok", hl=[3])
rr = res.copy(); table_png(pd.DataFrame({"": rr.index, "pontosság": [f3(v) for v in rr.accuracy], "kiegyens. pont.": [f3(v) for v in rr["balanced acc"]], "recall": [f3(v) for v in rr.recall],
    "precision": [f3(v) for v in rr.precision], "PR-AUC": [f3(v) for v in rr["PR-AUC"]]}), "tbl_vegso", hl=[1])
fig, ax = plt.subplots(figsize=(10.5, 4.8))
for col, colr in [("accuracy", INK2), ("balanced acc", BLUE), ("recall", ORANGE)]:
    ax.plot(ts, [metrics(y_train, p_oof, t)[col] for t in ts], color=colr, lw=2.8, label={"accuracy": "pontosság", "balanced acc": "kiegyensúlyozott pontosság", "recall": "recall"}[col])
t_star0 = max(ts, key=lambda t: balanced_accuracy_score(y_train, p_oof > t))
ax.axvline(t_acc, color=INK2, ls="--", lw=1.2); ax.axvline(t_star0, color=BLUE, ls="--", lw=1.2)
ax.text(t_acc + .01, .98, f"a pontosság csúcsa\n{hu(t_acc)}-nál", color=INK2, fontsize=14, va="top"); ax.text(t_star0 + .01, .5, f"kiegyens. pont.\ncsúcsa {hu(t_star0)}-nál", color=BLUE, fontsize=14, va="top")
ax.set_xlabel("döntési küszöb"); ax.set_ylim(0, 1.0); ax.legend(frameon=False, loc="center right"); hu_axes(ax); keep(fig, "loop_kuszob_gorbek"); NUMS["t_star_baseline"] = float(t_star0)
vals = [("tisztességes alap", log["alap xgboost @0,5"]["PR-AUC"], BLUE), ("előbb mintavétel, aztán CV", leaky["PR-AUC"], ORANGE), ("mintavétel a foldon belül", log["felülmintavételezés a foldon belül"]["PR-AUC"], BLUE)]
fig, ax = plt.subplots(figsize=(11, 4.6)); ax.bar([v[0] for v in vals], [v[1] for v in vals], color=[v[2] for v in vals], width=.55)
for i, v in enumerate(vals): ax.text(i, v[1] + .015, hu(v[1]), ha="center", fontsize=19, color=INK)
ax.set_ylim(0, 1); ax.set_ylabel("keresztvalidált PR-AUC"); ax.grid(axis="x", visible=False); hu_axes(ax, x=False); keep(fig, "loop_szivargas")
Path("numbers.json").write_text(json.dumps(NUMS, indent=1, default=float)); print("kész:", sorted(p.name for p in OUT.glob("*.png")))

kész: ['cm_lr.png', 'cm_vegso.png', 'cm_xgb.png', 'cv_abra.png', 'eda_ar.png', 'eda_hianyzas.png', 'eda_kor.png', 'final_pr_gorbe.png', 'ladder_folds.png', 'loop_arvaltozas.png', 'loop_kuszob_gorbek.png', 'loop_szivargas.png', 'metrics_roc_pr.png', 'tbl_hiba1.png', 'tbl_hiba2.png', 'tbl_jellemzok.png', 'tbl_kuszob_lr.png', 'tbl_letra.png', 'tbl_onehot.png', 'tbl_sorszam.png', 'tbl_tevedesek.png', 'tbl_utazas.png', 'tbl_vegso.png', 'tuning_elotte_utana.png', 'xgb_fontossag.png']
